In [83]:
# Se cargan las bibliotecas de pandas, numpy y matplotlib
import pandas as pd 
from pandas import ExcelWriter
from pandas import ExcelFile
import numpy as np
import matplotlib. pyplot as plt
from scipy import stats
from scipy.stats import anderson, normaltest, probplot, t
import time

print("Bibliotecas importadas correctamente")

Bibliotecas importadas correctamente


In [84]:
#Decodifiacion de Hamming (7,4)
#bits_rx es la lista de bits recibida del demodulador con ruido

def hamming_7_4_decode(bits_rx):
    
    bits_rx = np.array(bits_rx, dtype=int)           #lo convierte a un array de ints
    total_blocks = len(bits_rx) // 7

    bits_rx = bits_rx[:total_blocks * 7].reshape((total_blocks, 7))

    decoded = []
    corrected_blocks = 0

    for c in bits_rx:
        # Calculo del síndrome 
        s1 = c[0] ^ c[2] ^ c[4] ^ c[6]  # paridad de las posiciones 1,3,5,7
        s2 = c[1] ^ c[2] ^ c[5] ^ c[6]  # paridad de las posiciones 2,3,6,7
        s3 = c[3] ^ c[4] ^ c[5] ^ c[6]  # paridad de las posiciones 4,5,6,7

        S = s1 + 2 * s2 + 4 * s3    # síndrome 

        if S != 0:         #si el sindrome es = 0 no se detecta error, si sindrome != 0 hay un error en la posicion S-1 (S-1 porque Python inicia en pos 0)
            c[S - 1] ^= 1  # corregir bit 
            corrected_blocks += 1

        # Extraer los bits de datos que estan en las posiciones 3,5,6,7, entonces índices 2,4,5,6 
        #Ignora los bits de paridad
        decoded.extend([c[2], c[4], c[5], c[6]])

    return np.array(decoded, dtype=int), corrected_blocks, total_blocks


In [85]:
#Decodifiacion de Hamming (15,11)
#bits_rx es la lista de bits recibida del demodulador con ruido

def hamming_15_11_decode(bits_rx):

    bits_rx = np.array(bits_rx, dtype=int)
    total_blocks = len(bits_rx) // 15

    bits_rx = bits_rx[:total_blocks * 15].reshape((total_blocks, 15))

    decoded = []
    corrected_blocks = 0

    for c in bits_rx:
        # Calculo del síndrome de Hamming(15,11)
        s1 = c[0] ^ c[2] ^ c[4] ^ c[6] ^ c[8] ^ c[10] ^ c[12] ^ c[14]    # paridad de las posiciones 1,3,5,7,9,11,13,15
        s2 = c[1] ^ c[2] ^ c[5] ^ c[6] ^ c[9] ^ c[10] ^ c[13] ^ c[14]    # paridad de las posiciones 2,3,6,7,10,11,14,15
        s4 = c[3] ^ c[4] ^ c[5] ^ c[6] ^ c[11] ^ c[12] ^ c[13] ^ c[14]   # paridad de las posiciones 4,5,6,7,12,13,14,15
        s8 = c[7] ^ c[8] ^ c[9] ^ c[10] ^ c[11] ^ c[12] ^ c[13] ^ c[14]  # paridad de las posiciones 8,9,10,11,12,13,14,15

        S = s1 + 2 * s2 + 4 * s4 + 8 * s8  

        if S != 0:
            c[S - 1] ^= 1  
            corrected_blocks += 1

        # Extraer datos: posiciones 3,5,6,7,9,10,11,12,13,14,15
        data_idx = [2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14]
        decoded.extend(c[data_idx])

    return np.array(decoded, dtype=int), corrected_blocks, total_blocks

In [86]:
#Funcion para pasar de bits a bytes 

def bits_to_bytes(bits):
    
    bits = np.array(bits, dtype=int)
    num_bytes = len(bits) // 8
    bits = bits[:num_bytes*8].reshape((num_bytes, 8))

    values = []
    for byte in bits:
        v = 0
        for b in byte:
            v = (v << 1) | int(b)
        values.append(v)
    return np.array(values, dtype=int)


In [87]:
#Funcion para pasar de ints a bits

def ints_to_bits8(integers):
    bits = []
    for v in integers:
        b = bin(int(v))[2:].zfill(8)
        bits.extend(int(x) for x in b)
    return np.array(bits, dtype=int)


In [88]:
#Funcion para calcular el BER entre dos vectores de bits
def compute_ber(bits_ref, bits_rec):
    L = min(len(bits_ref), len(bits_rec))
    if L == 0:
        return 0.0, 0, 0
    ref = np.array(bits_ref[:L], dtype=int)
    rec = np.array(bits_rec[:L], dtype=int)
    errores = np.sum(ref != rec)
    ber = errores / L
    return ber, errores, L

def log_result(filename, text_line):
    with open(filename, 'a') as f:
        f.write(text_line + '\n')


In [89]:
def ejecutar_decodificacion_una_vez(snr_db,
                                    ruta_dataset,
                                    ruta_demod_74,
                                    ruta_demod_1511,
                                    log_74,
                                    log_1511):
    # 4.1 Leer BPM originales
    datos = pd.read_excel(ruta_dataset)
    BPM = datos['BPM'].round().astype(int)
    bits_originales = ints_to_bits8(BPM)

    
    # 4.2 Leer bits demodulados Hamming(7,4)
    with open(ruta_demod_74, "r") as f:
        raw = f.read()
    
    # 2. Limpiar caracteres no numéricos
    raw = raw.replace('[', ' ')
    raw = raw.replace(']', ' ')
    raw = raw.replace(',', ' ')
    
    # 3. Convertir la cadena en un vector de int (0/1)
    bits_rx_74 = np.fromstring(raw, dtype=int, sep=' ')

    # Decodificar Hamming(7,4)
    dec_bits_74, corr_74, total_blocks_74 = hamming_7_4_decode(bits_rx_74)

    # Reconstruir BPM (bytes) desde los bits decodificados
    bpm_rec_74 = bits_to_bytes(dec_bits_74)
    np.savetxt("Datos_decodificados/BPM_decodificados_7_4.txt", bpm_rec_74, fmt="%d")

    # Para BER a nivel de bit, volvemos a bits
    bits_rec_74 = ints_to_bits8(bpm_rec_74)

    ber74, err74, total_bits74 = compute_ber(bits_originales, bits_rec_74)


    
    # 4.3 Leer bits demodulados Hamming(15,11)
    #demod_1511 = np.loadtxt(ruta_demod_1511)

    with open(ruta_demod_1511, "r") as f:
        raw = f.read()
    
    # 2. Limpiar caracteres no numéricos
    raw = raw.replace('[', ' ')
    raw = raw.replace(']', ' ')
    raw = raw.replace(',', ' ')
    
    # 3. Convertir la cadena en un vector de int (0/1)
    bits_rx_1511 = np.fromstring(raw, dtype=int, sep=' ')

    # Decodificar Hamming(15,11)
    dec_bits_1511, corr_1511, total_blocks_1511 = hamming_15_11_decode(bits_rx_1511)

    # Bloques de 11 bits → nos interesan los primeros 8 bits de cada bloque
    dec_bits_1511 = dec_bits_1511.reshape((-1, 11))
    payload_bits = dec_bits_1511[:, :8].reshape(-1)

    bpm_rec_1511 = bits_to_bytes(payload_bits)
    np.savetxt("Datos_decodificados/BPM_decodificados_15_11.txt", bpm_rec_1511, fmt="%d")
    bits_rec_1511 = ints_to_bits8(bpm_rec_1511)

    ber1511, err1511, total_bits1511 = compute_ber(bits_originales, bits_rec_1511)

    # 4.4 Log de resultados con SNR
    linea74 = (f"SNR_dB={snr_db}, esquema=Hamming(7,4), BER={ber74:.6e}, "
               f"errores={err74}, bits={total_bits74}, "
               f"bloques_corregidos={corr_74}, total_bloques={total_blocks_74}")
    log_result(log_74, linea74)

    linea1511 = (f"SNR_dB={snr_db}, esquema=Hamming(15,11), BER={ber1511:.6e}, "
                 f"errores={err1511}, bits={total_bits1511}, "
                 f"bloques_corregidos={corr_1511}, total_bloques={total_blocks_1511}")
    log_result(log_1511, linea1511)

    return (ber74, ber1511)


In [90]:
def medir_tiempo_promedio(snr_db,
                          ruta_dataset="Dataset.xlsx",
                          ruta_demod_74="Datos_codificados/codificacion_hamming_7_4.txt",
                          ruta_demod_1511="Datos_codificados/codificacion_hamming_15_11.txt",
                          log_74="Datos_decodificados/BER_log_7_4.txt",
                          log_1511="Datos_decodificados/BER_log_15_11.txt",
                          repeticiones=200):
    t_total = 0.0
    for _ in range(repeticiones):
        t_ini = time.time()
        ejecutar_decodificacion_una_vez(snr_db,
                                        ruta_dataset,
                                        ruta_demod_74,
                                        ruta_demod_1511,
                                        log_74,
                                        log_1511)
        t_fin = time.time()
        t_total += (t_fin - t_ini)
    t_prom = t_total / repeticiones
    print(f"Tiempo promedio de ejecución del decodificador (Hamming 7,4 y 15,11) = {t_prom:.6f} s")
    return t_prom


In [91]:
# Llamada a todas las funciones

tiempo_promedio = medir_tiempo_promedio(snr_db=10.0)


Tiempo promedio de ejecución del decodificador (Hamming 7,4 y 15,11) = 0.016417 s
